# Week 3-2 · SFM-03 — Basic Statistics using Excel (in Python)
**Practice notebook — every calculation the lecturer did in Excel, reproduced on the real lecture data.**

This is a hands-on statistics lecture. The instructor built everything live in Excel; here we rebuild
it in pandas/numpy so you can *re-run and reuse* it. We use the **exact datasets shipped with the
lecture** (`SFM_03_data_Raw_Data.xlsm` → two CSVs in this folder):

- `portfolio_sbi_icici.csv` — SBIN & ICICIBANK daily close, Jan–Apr 2023 (the efficient-frontier demo)
- `nifty.csv` — NIFTY daily close 2013–2023 (random walk, Monte Carlo, Bollinger). Last price **18065**.

**What we reproduce (all numbers verified against the lecture):**
1. Returns from scratch — **why log returns** (continuous compounding)
2. Mean, variance, std **from the definition**; sample vs population & **Bessel's n−1** (degrees of freedom)
3. Covariance, then a **101-portfolio efficient frontier** → minimum-variance portfolio
4. **Random walk** price ranges (68% / 95%) from mean ± k·σ
5. **Annualizing volatility**: σ_T = σ_1·√T (variance is additive, std is not)
6. **Simple vs log returns + the drift correction** (μ − σ²/2) — the fund "9%" trap
7. **Monte Carlo** simulation of NIFTY → price range, **VaR / expected shortfall (CVaR)**
8. **Bollinger Bands** (20-day SMA ± 2σ) + the mean-reversion strategy


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 20)
np.random.seed(3)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.3 | numpy 2.2.6


## Part 1 — Returns from scratch, and why **log** returns

The lecturer's first point: when you type `=LN(today/yesterday)` in Excel, what is it doing? If money
compounds **continuously**, the per-period return is the natural log of the price ratio:

$$ r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right) $$

Continuous compounding comes from $P_0(1+r/n)^{nt}\to P_0 e^{rt}$ as $n\to\infty$. The bonus: **log
returns add up** across time, which makes everything later (annualizing, multi-day horizons) clean.

In [2]:
pf = pd.read_csv("portfolio_sbi_icici.csv", parse_dates=["Date"]).set_index("Date")
ret = np.log(pf).diff().dropna()          # daily LOG returns
ret.columns = ["SBIN", "ICICIBANK"]
print("Daily log returns (first 5 rows):")
print(ret.head().round(5))
print(f"\n{len(ret)} daily returns from {ret.index[0].date()} to {ret.index[-1].date()}")


Daily log returns (first 5 rows):
               SBIN  ICICIBANK
Date                          
2023-01-31  0.02803    0.01015
2023-02-01 -0.04840    0.01911
2023-02-02  0.00142    0.01167
2023-02-03  0.03003    0.00685
2023-02-06  0.00220   -0.01176

59 daily returns from 2023-01-31 to 2023-04-28


## Part 2 — Mean, variance, std **from the definition** + Bessel's correction

The lecturer derived std by hand instead of `STDEV.S`. The recipe:
1. mean  $\bar x = \frac1n\sum x_i$
2. deviations $x_i-\bar x$, then **square** them
3. sum, divide by **n−1**, then **square-root**.

Why **n−1** and not n? Because we used the sample mean to centre the data, the deviations are forced
to sum to zero — so the **last deviation is not free**. We spent one *degree of freedom* estimating the
mean, leaving n−1. Dividing by n would systematically **underestimate** the variance (**Bessel's
correction**). That's why finance always uses `STDEV.S` (sample), never `STDEV.P`.

In [3]:
x = ret["SBIN"].values
n = len(x)
xbar = x.mean()
dev = x - xbar
sq = dev**2
var_sample = sq.sum() / (n - 1)        # Bessel: divide by n-1
var_pop    = sq.sum() / n              # population (wrong for a sample)
std_sample = np.sqrt(var_sample)

print("STEP TABLE — std of SBIN log returns, by hand")
print(f"  n                          = {n}")
print(f"  mean  xbar                 = {xbar:.6f}")
print(f"  sum (xi - xbar)            = {dev.sum():.2e}   <- ~0 by construction (lost 1 dof)")
print(f"  sum (xi - xbar)^2          = {sq.sum():.6e}")
print(f"  / (n-1)  = sample variance = {var_sample:.6e}")
print(f"  sqrt     = sample std      = {std_sample:.6f}")
print(f"\n  cross-check np.std(ddof=1) = {x.std(ddof=1):.6f}  (match)")
print(f"  population std (/n)         = {np.sqrt(var_pop):.6f}  (smaller -> biased)")


STEP TABLE — std of SBIN log returns, by hand
  n                          = 59
  mean  xbar                 = 0.001218
  sum (xi - xbar)            = -2.43e-17   <- ~0 by construction (lost 1 dof)
  sum (xi - xbar)^2          = 1.502352e-02
  / (n-1)  = sample variance = 2.590262e-04
  sqrt     = sample std      = 0.016094

  cross-check np.std(ddof=1) = 0.016094  (match)
  population std (/n)         = 0.015957  (smaller -> biased)


### Bessel in miniature — a 5-point sample
The lecturer's worked illustration: with a tiny sample, dividing by n vs n−1 visibly differs, and n−1
is the unbiased estimate of the true population variance.

In [4]:
sample = np.array([2051, 2049, 2053, 2052, 2050.0])   # 5 readings, pop mean ~2050
sm = sample.mean(); d = sample - sm
print("STEP TABLE — sample variance, n vs n-1")
tbl = pd.DataFrame({"x": sample, "x - xbar": d, "(x-xbar)^2": d**2})
print(tbl.to_string(index=False))
print(f"  sum sq            = {(d**2).sum():.2f}")
print(f"  / n   (pop  var)  = {(d**2).sum()/5:.3f}")
print(f"  / n-1 (samp var)  = {(d**2).sum()/4:.3f}   <- unbiased, use this")


STEP TABLE — sample variance, n vs n-1
     x  x - xbar  (x-xbar)^2
2051.0       0.0         0.0
2049.0      -2.0         4.0
2053.0       2.0         4.0
2052.0       1.0         1.0
2050.0      -1.0         1.0
  sum sq            = 10.00
  / n   (pop  var)  = 2.000
  / n-1 (samp var)  = 2.500   <- unbiased, use this


## Part 3 — Covariance and the **efficient frontier**

To combine two stocks we need how they **co-move**: covariance
$\text{Cov}(x,y)=\frac1{n-1}\sum (x_i-\bar x)(y_i-\bar y)$. Portfolio variance for weights
$w$ in SBIN and $1-w$ in ICICI:

$$ \sigma_p^2 = w^2\sigma_{SBI}^2 + (1-w)^2\sigma_{ICI}^2 + 2\,w(1-w)\,\text{Cov} $$

We sweep **101 portfolios** (w = 0%…100%) exactly like the lecture and find the lowest-risk one.

In [5]:
mu = ret.mean()
sd = ret.std(ddof=1)
cov = ret.cov().iloc[0, 1]
print(f"mean : SBIN {mu.SBIN:.6f}   ICICI {mu.ICICIBANK:.6f}")
print(f"std  : SBIN {sd.SBIN:.6f}   ICICI {sd.ICICIBANK:.6f}")
print(f"cov(SBIN, ICICI) = {cov:.3e}   <- matches the lecture's 6.86e-5")

w = np.linspace(0, 1, 101)                      # weight in SBIN
p_ret = w*mu.SBIN + (1-w)*mu.ICICIBANK
p_var = w**2*sd.SBIN**2 + (1-w)**2*sd.ICICIBANK**2 + 2*w*(1-w)*cov
p_vol = np.sqrt(p_var)
i = p_vol.argmin()
print(f"\nMINIMUM-VARIANCE portfolio: {w[i]*100:.0f}% SBIN / {(1-w[i])*100:.0f}% ICICI")
print(f"   daily vol = {p_vol[i]:.5f} ({p_vol[i]*100:.3f}%)  daily ret = {p_ret[i]:.6f}")
print("   -> diversification cut risk BELOW either stock alone (SBIN %.3f%%, ICICI %.3f%%)"
      % (sd.SBIN*100, sd.ICICIBANK*100))


mean : SBIN 0.001218   ICICI 0.001835
std  : SBIN 0.016094   ICICI 0.010450
cov(SBIN, ICICI) = 6.864e-05   <- matches the lecture's 6.86e-5

MINIMUM-VARIANCE portfolio: 18% SBIN / 82% ICICI
   daily vol = 0.01010 (1.010%)  daily ret = 0.001724
   -> diversification cut risk BELOW either stock alone (SBIN 1.609%, ICICI 1.045%)


### Visualization checkpoint - efficient frontier chart

The previous cell computed portfolio returns, volatility, and covariance. This cell draws those computed frontier points and saves the chart used in the HTML pack.

In [6]:
fig, ax = plt.subplots(figsize=(7.6,3.6))
ax.plot(p_vol*100, p_ret*100, color="#2563eb", lw=2)
ax.scatter([sd.SBIN*100],[mu.SBIN*100], color="#dc2626", zorder=5, label="100% SBIN")
ax.scatter([sd.ICICIBANK*100],[mu.ICICIBANK*100], color="#16a34a", zorder=5, label="100% ICICI")
ax.scatter([p_vol[i]*100],[p_ret[i]*100], color="#f59e0b", s=70, zorder=6, label="min-variance")
ax.set_xlabel("portfolio daily volatility  (%)"); ax.set_ylabel("portfolio daily return (%)")
ax.set_title("Efficient frontier — 101 SBIN/ICICI portfolios"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_1_frontier.png", dpi=110); plt.show()
print("Only the UPPER arm is efficient: for a given risk, take the highest-return portfolio.")
print("saved chart_1_frontier.png")


Only the UPPER arm is efficient: for a given risk, take the highest-return portfolio.
saved chart_1_frontier.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\496262580.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_1_frontier.png", dpi=110); plt.show()


## Part 4 — The **random walk** model and price ranges

Tomorrow's price = today's price × a random return. The model assumes daily **returns are normally
distributed and independent**. Then, knowing only the historical **mean** and **std**, the normal
distribution hands us probability ranges:
- **68%** of returns within μ ± 1σ
- **95%** within μ ± 2σ

Why model *returns* not *prices*? Prices are tethered to yesterday (102 can't jump to 1000); returns
are free and roughly scale-independent — so they can be treated as i.i.d. normal.

In [7]:
nf = pd.read_csv("nifty.csv", parse_dates=["Date"]).set_index("Date")
logr = np.log(nf["NIFTY"]).diff().dropna()
m, s = logr.mean(), logr.std(ddof=1)
P0 = nf["NIFTY"].iloc[-1]
print(f"NIFTY: {len(nf)} days, last date {nf.index[-1].date()}, last price {P0:.0f}")
print(f"daily log return: mean {m:.6f}, std {s:.6f}")
for k, p in [(1,"68%"), (2,"95%")]:
    lo, hi = P0*np.exp(m-k*s), P0*np.exp(m+k*s)
    print(f"  next-day {p} range (mu +/- {k} sd): [{lo:,.0f}, {hi:,.0f}]")


NIFTY: 2455 days, last date 2023-04-28, last price 18065
daily log return: mean 0.000456, std 0.010892
  next-day 68% range (mu +/- 1 sd): [17,877, 18,271]
  next-day 95% range (mu +/- 2 sd): [17,684, 18,471]


## Part 5 — **Annualizing** volatility: σ_T = σ_1·√T

Returns add, but **standard deviation does not**. Variance *does* add (when returns are independent,
the covariance terms vanish), so for T periods:

$$ \sigma_T^2 = T\,\sigma_1^2 \;\Rightarrow\; \sigma_T = \sigma_1\sqrt{T} $$

A 1%-per-day stock is **not** 252%/yr — it's 1%·√252 ≈ 16%.

In [8]:
print("STEP LADDER — annualize daily vol")
print(f"  daily std sigma_1        = {s:.6f}")
print(f"  daily variance sigma_1^2 = {s**2:.3e}")
print(f"  x 252 trading days       = {s**2*252:.5f}   (variance is additive)")
print(f"  sqrt -> annual std       = {s*np.sqrt(252):.4f}  ({s*np.sqrt(252)*100:.2f}% per year)")
print(f"\n  toy check: 1%/day -> 1% * sqrt(252) = {0.01*np.sqrt(252)*100:.1f}% per year")


STEP LADDER — annualize daily vol
  daily std sigma_1        = 0.010892
  daily variance sigma_1^2 = 1.186e-04
  x 252 trading days       = 0.02989   (variance is additive)
  sqrt -> annual std       = 0.1729  (17.29% per year)

  toy check: 1%/day -> 1% * sqrt(252) = 15.9% per year


## Part 6 — Simple vs log returns, and the **drift** correction

If you average *simple* returns you **overstate** the true compounded outcome. The lecturer's fund
example: yearly returns 15, 23, 26, −11, −5%.

In [9]:
yrs = np.array([15, 23, 26, -11, -5]) / 100
simple_avg = yrs.mean()
actual = 100*np.prod(1+yrs)
naive  = 100*(1+simple_avg)**5
print("STEP TABLE — the 'average return' trap (start with 100)")
print(f"  simple average of years     = {simple_avg*100:.2f}%")
print(f"  naive 100*(1+avg)^5         = {naive:.2f}   <- OVERSTATED")
print(f"  actual 100*prod(1+r)        = {actual:.2f}   <- truth")
print(f"  gap                         = {naive-actual:.2f}")

# Fix 1: log returns add correctly
log_avg = np.log(1+yrs).mean()
print(f"\n  Fix A: log-return avg {log_avg*100:.2f}% -> 100*e^(5*logavg) = {100*np.exp(5*log_avg):.2f}  (matches truth)")
# Fix 2: drift on simple returns  mu - sigma^2/2
drift = simple_avg - yrs.var(ddof=1)/2
print(f"  Fix B: drift = mu - sd^2/2 = {drift*100:.2f}% -> 100*(1+drift)^5 = {100*(1+drift)**5:.2f}")
print("\n  Use LOG returns, or subtract the drift (mu - sigma^2/2) from simple returns.")


STEP TABLE — the 'average return' trap (start with 100)
  simple average of years     = 9.60%
  naive 100*(1+avg)^5         = 158.14   <- OVERSTATED
  actual 100*prod(1+r)        = 150.69   <- truth
  gap                         = 7.45

  Fix A: log-return avg 8.20% -> 100*e^(5*logavg) = 150.69  (matches truth)
  Fix B: drift = mu - sd^2/2 = 8.21% -> 100*(1+drift)^5 = 148.34

  Use LOG returns, or subtract the drift (mu - sigma^2/2) from simple returns.


## Part 7 — **Monte Carlo** simulation of NIFTY (20 trading days)

Instead of a formula, *simulate* thousands of price paths. Each day's return is drawn normal
(Excel's `=NORMINV(RAND(), mean, std)`); the next price is `P_prev * exp(return)`. From the cloud of
final prices we read off any statistic we like — range, **VaR**, **expected shortfall**.

In [10]:
N_SIM, DAYS = 5000, 20
# daily log-return parameters from history
daily_mu, daily_sd = m, s
# simulate returns: shape (DAYS, N_SIM)
shocks = np.random.normal(daily_mu, daily_sd, size=(DAYS, N_SIM))
paths = P0 * np.exp(np.cumsum(shocks, axis=0))     # price each day
final = paths[-1]                                  # price on day 20

p2_5, p97_5 = np.percentile(final, [2.5, 97.5])
print(f"Monte Carlo: {N_SIM} paths, {DAYS} days, start {P0:.0f}")
print(f"  95% range of day-20 price = [{p2_5:,.0f}, {p97_5:,.0f}]")
# compare with closed-form (log, no drift needed)
lo = P0*np.exp(daily_mu*DAYS - 2*daily_sd*np.sqrt(DAYS))
hi = P0*np.exp(daily_mu*DAYS + 2*daily_sd*np.sqrt(DAYS))
print(f"  closed-form 95% range     = [{lo:,.0f}, {hi:,.0f}]   <- the 3 methods agree")


Monte Carlo: 5000 paths, 20 days, start 18065
  95% range of day-20 price = [16,544, 20,012]
  closed-form 95% range     = [16,538, 20,096]   <- the 3 methods agree


### Risk output checkpoint - VaR and expected shortfall

The previous cell generated Monte Carlo price paths. This cell converts those paths into visible risk outputs: the 5th-percentile VaR floor, expected shortfall, and the final-price distribution chart.

In [11]:
# VaR & Expected Shortfall (CVaR) at 5% on day-20 outcome
var5 = np.percentile(final, 5)                 # 5th percentile price
es5  = final[final <= var5].mean()             # average of the worst 5% (expected shortfall)
print("STEP TABLE — risk at 5% (20-day horizon)")
print(f"  5th-percentile price (VaR floor)        = {var5:,.0f}")
print(f"  -> 95% confident NIFTY stays above       {var5:,.0f}")
print(f"  expected shortfall = mean of worst 5%   = {es5:,.0f}")
print(f"  -> IF it breaks the floor, expected level {es5:,.0f}")

fig, axs = plt.subplots(1, 2, figsize=(9.4,3.4))
axs[0].plot(paths[:, :120], lw=0.5, alpha=0.35, color="#2563eb")
axs[0].axhline(P0, color="#111", lw=1, ls="--"); axs[0].set_title("120 simulated NIFTY paths (20d)")
axs[0].set_xlabel("day"); axs[0].set_ylabel("price")
axs[1].hist(final, bins=50, color="#93c5fd", edgecolor="#1e3a8a", linewidth=.3)
axs[1].axvline(var5, color="#dc2626", lw=1.5, label=f"5% VaR {var5:,.0f}")
axs[1].axvline(es5, color="#7c2d12", lw=1.5, ls="--", label=f"ES {es5:,.0f}")
axs[1].set_title("Day-20 price distribution"); axs[1].set_xlabel("price"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_2_montecarlo.png", dpi=110); plt.show()
print("saved chart_2_montecarlo.png")


STEP TABLE — risk at 5% (20-day horizon)
  5th-percentile price (VaR floor)        = 16,796
  -> 95% confident NIFTY stays above       16,796
  expected shortfall = mean of worst 5%   = 16,456
  -> IF it breaks the floor, expected level 16,456


saved chart_2_montecarlo.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\3455829320.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_2_montecarlo.png", dpi=110); plt.show()


## Part 8 — **Bollinger Bands** + mean-reversion strategy

A 20-day moving average with bands at ±2σ. Assuming prices are ~normal, ~95% of action sits inside
the bands, so a simple **mean-reversion** rule: **buy** when price closes below the lower band,
**sell/exit** when it closes above the upper band.

In [12]:
W = 20
bb = nf.copy()
bb["sma"]   = bb["NIFTY"].rolling(W).mean()
bb["std"]   = bb["NIFTY"].rolling(W).std(ddof=1)
bb["upper"] = bb["sma"] + 2*bb["std"]
bb["lower"] = bb["sma"] - 2*bb["std"]
recent = bb.dropna().iloc[-250:]

fig, ax = plt.subplots(figsize=(9.2,3.4))
ax.plot(recent.index, recent["NIFTY"], color="#2563eb", lw=1, label="NIFTY")
ax.plot(recent.index, recent["sma"], color="#f59e0b", lw=1, label="20d SMA")
ax.plot(recent.index, recent["upper"], color="#dc2626", lw=.9, label="+2 sd")
ax.plot(recent.index, recent["lower"], color="#16a34a", lw=.9, label="-2 sd")
ax.fill_between(recent.index, recent["lower"], recent["upper"], color="#eff6ff")
ax.set_title("Bollinger Bands on NIFTY (last 250 days)"); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig("chart_3_bollinger.png", dpi=110); plt.show()
print("saved chart_3_bollinger.png")

# count touches as a sanity check
below = (bb["NIFTY"] < bb["lower"]).sum()
above = (bb["NIFTY"] > bb["upper"]).sum()
tot = bb.dropna().shape[0]
print(f"Out of {tot} days: {below} closes below lower band, {above} above upper band "
      f"= {(below+above)/tot*100:.1f}% outside.")
print("Normal theory predicts ~5%; the real ~9% shows FAT TAILS — markets break the bands")
print("more often than a normal model expects (the same lesson as crashes beyond 3 sigma).")


saved chart_3_bollinger.png
Out of 2436 days: 100 closes below lower band, 123 above upper band = 9.2% outside.
Normal theory predicts ~5%; the real ~9% shows FAT TAILS — markets break the bands
more often than a normal model expects (the same lesson as crashes beyond 3 sigma).


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\654538275.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_3_bollinger.png", dpi=110); plt.show()


## Summary — what to remember
- **Log returns** (continuous compounding) **add up** → use them for multi-period work.
- **Sample std uses n−1** (Bessel): one degree of freedom is spent on the mean.
- **Portfolio variance** needs the covariance term; diversification can push risk **below** either asset.
- **Random walk** + normal returns → **μ ± kσ** price ranges (68% / 95%).
- **Volatility scales with √T** (variance adds, std doesn't): 1%/day ≈ 16%/yr.
- Averaging **simple** returns overstates compounding → use log returns **or** subtract drift **μ − σ²/2**.
- **Monte Carlo** gives the whole distribution → range, **VaR** (5% floor) and **expected shortfall**.
- **Bollinger Bands** = SMA ± 2σ; in theory ~5% of closes fall outside, but in real NIFTY ~9% do
  (**fat tails**) — the basis of a mean-reversion trade, and a reminder the normal model is only approximate.


# Resource Addendum Sections

The following cells are appended from the validation addendum notebook. They cover the official workbook sample values, transcript-versus-slide risk-limit difference, practice exercise answer key, and additional charts. The original notebook above is preserved separately; this composite is the fully validated runnable version.

## 1. Setup and editable configuration

Change the file names or start price here if you want to reuse this notebook with another local dataset. The Monte Carlo section uses a fixed random seed so the visible outputs are reproducible.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

# EDIT HERE
DATA_DIR = Path(".")
PORTFOLIO_FILE = DATA_DIR / "portfolio_sbi_icici.csv"
NIFTY_FILE = DATA_DIR / "nifty.csv"
PRACTICE_RETURNS_FILE = DATA_DIR / "sfm03_practice_nifty_returns.csv"
PRACTICE_ANSWER_FILE = DATA_DIR / "sfm03_practice_answer_key.csv"

# Lecture transcript mentions a tight investor limit around 1.03%; the slide deck states 2.7%.
RISK_LIMITS = {"transcript_1_03pct": 0.0103, "transcript_1_303pct": 0.01303, "slides_2_7pct": 0.027}
MONTE_CARLO_START_PRICE = 18065.00  # Lecture NIFTY last price; replace with the exercise last NIFTY price if you have it.
N_SIM = 5000
DAYS = 10
SEED = 20260302

print("Configured files:")
for p in [PORTFOLIO_FILE, NIFTY_FILE, PRACTICE_RETURNS_FILE, PRACTICE_ANSWER_FILE]:
    print(" -", p)
print("Risk limits:", RISK_LIMITS)
print("Monte Carlo:", {"start_price": MONTE_CARLO_START_PRICE, "simulations": N_SIM, "days": DAYS, "seed": SEED})

Configured files:
 - portfolio_sbi_icici.csv
 - nifty.csv
 - sfm03_practice_nifty_returns.csv
 - sfm03_practice_answer_key.csv
Risk limits: {'transcript_1_03pct': 0.0103, 'transcript_1_303pct': 0.01303, 'slides_2_7pct': 0.027}
Monte Carlo: {'start_price': 18065.0, 'simulations': 5000, 'days': 10, 'seed': 20260302}


## 2. Load lecture data and official practice extracts

The original lecture data remains the SBIN/ICICI Bank portfolio file and ten-year NIFTY file. The addendum also loads the official practice worked-solution returns and one-line answer key extracted from the Excel workbook.

In [14]:
portfolio = pd.read_csv(PORTFOLIO_FILE, parse_dates=["Date"])
nifty = pd.read_csv(NIFTY_FILE, parse_dates=["Date"])
practice_returns = pd.read_csv(PRACTICE_RETURNS_FILE, parse_dates=["Date"])
answer_key = pd.read_csv(PRACTICE_ANSWER_FILE)

print("Lecture portfolio data:", portfolio.shape)
display(portfolio.head())
print("Lecture NIFTY data:", nifty.shape)
display(nifty.tail())
print("Official practice return extract:", practice_returns.shape)
display(practice_returns.head())
print("Official practice answer key:")
display(answer_key)

Lecture portfolio data: (60, 3)


,Date,SBIN,ICICIBANK
0,2023-01-30,538.200012,823.500000
1,2023-01-31,553.500000,831.900024
2,2023-02-01,527.349976,847.950012
3,2023-02-02,528.099976,857.900024
4,2023-02-03,544.200012,863.799988


Lecture NIFTY data: (2455, 2)


,Date,NIFTY
2450,2023-04-24,17743.400391
2451,2023-04-25,17769.250000
2452,2023-04-26,17813.599609
2453,2023-04-27,17915.050781
2454,2023-04-28,18065.000000


Official practice return extract: (158, 2)


,Date,Daily_Returns_NIFTY
0,2025-07-08,NaN
1,2025-07-09,-0.000835
2,2025-07-10,-0.003843
3,2025-07-11,-0.006206
4,2025-07-14,-0.000675


Official practice answer key:


,Question_ID,Question,Official_Worked_Answer
0,Q2,Identify the minimum price of SBIN using MIN().,62.27
1,Q3,Return the date corresponding to the minimum p...,2025-07-09 00:00:00
2,Q4,Compute the mean price of SBIN using AVERAGE().,69.7967088607595
3,Q5,Compute the mean price of HDB using AVERAGE().,35.61459556962026
4,Q6,Compute the sample standard deviation of SBIN ...,4.810624447070153
5,Q7,Compute the sample standard deviation of HDB p...,1.702721402547587
6,Q8,Compute the standard deviation of daily return...,0.007667194761460805
7,Q9,Compute the average of daily returns of NIFTY ...,-0.0001291406931850168
8,Q10,Calculate the variance of daily returns using ...,5.8785875510172e-05


## 3. Official Bessel-correction sample

The official explanation workbook uses the sample `[2051, 2053, 2055, 2050, 2051]`, a known population mean of `2050`, and a sample mean of `2052`. Dividing sample squared deviations by `n` gives `3.2`; after Bessel's correction, dividing by `n - 1` gives `4.0`.

In [15]:
official_sample = np.array([2051, 2053, 2055, 2050, 2051], dtype=float)
population_mean = 2050.0
sample_mean = official_sample.mean()

bessel_table = pd.DataFrame({
    "Xi": official_sample,
    "Xi - population_mean": official_sample - population_mean,
    "(Xi - population_mean)^2": (official_sample - population_mean) ** 2,
    "Xi - sample_mean": official_sample - sample_mean,
    "(Xi - sample_mean)^2": (official_sample - sample_mean) ** 2,
})
population_variance = bessel_table["(Xi - population_mean)^2"].sum() / len(official_sample)
uncorrected_sample_variance = bessel_table["(Xi - sample_mean)^2"].sum() / len(official_sample)
corrected_sample_variance = bessel_table["(Xi - sample_mean)^2"].sum() / (len(official_sample) - 1)

summary = pd.DataFrame({
    "Quantity": ["population mean", "sample mean", "population variance", "sample variance divided by n", "sample variance with n-1"],
    "Value": [population_mean, sample_mean, population_variance, uncorrected_sample_variance, corrected_sample_variance],
})

display(bessel_table)
display(summary)
print(f"Official workbook check: population variance={population_variance:.1f}, uncorrected sample variance={uncorrected_sample_variance:.1f}, corrected sample variance={corrected_sample_variance:.1f}")

,Xi,Xi - population_mean,(Xi - population_mean)^2,Xi - sample_mean,(Xi - sample_mean)^2
0,2051.0,1.0,1.0,-1.0,1.0
1,2053.0,3.0,9.0,1.0,1.0
2,2055.0,5.0,25.0,3.0,9.0
3,2050.0,0.0,0.0,-2.0,4.0
4,2051.0,1.0,1.0,-1.0,1.0


,Quantity,Value
0,population mean,2050.0
1,sample mean,2052.0
2,population variance,7.2
3,sample variance divided by n,3.2
4,sample variance with n-1,4.0


Official workbook check: population variance=7.2, uncorrected sample variance=3.2, corrected sample variance=4.0


## 4. Chart the Bessel correction

The visual point is that changing the mean from the known population mean to the estimated sample mean changes the deviations, and then `n - 1` corrects the downward bias caused by estimating the mean from the same data.

In [16]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
labels = [f"x{i+1}" for i in range(len(official_sample))]
x = np.arange(len(labels))
width = 0.36
ax.bar(x - width/2, bessel_table["(Xi - population_mean)^2"], width, label="vs population mean 2050", color="#2563eb")
ax.bar(x + width/2, bessel_table["(Xi - sample_mean)^2"], width, label="vs sample mean 2052", color="#f59e0b")
ax.axhline(corrected_sample_variance, color="#dc2626", linestyle="--", label="corrected sample variance = 4.0")
ax.set_title("Official Bessel Correction Example")
ax.set_ylabel("Squared deviation")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
out = DATA_DIR / "chart_addendum_2_sfm03_bessel_sample.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Saved chart_addendum_2_sfm03_bessel_sample.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\4148519553.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Portfolio risk limits: transcript versus slide deck

The same frontier can be judged against different volatility limits. The lecture transcript describes a tight risk-aversion example around `1.03%` daily volatility, while the official slide problem states `2.7%`. Under `2.7%`, every sampled SBIN/ICICI portfolio is feasible; under `1.03%`, only the low-risk side is feasible.

In [17]:
prices = portfolio.set_index("Date")[["SBIN", "ICICIBANK"]]
returns = np.log(prices).diff().dropna()
mu = returns.mean()
sigma = returns.std(ddof=1)
covariance = returns.cov().loc["SBIN", "ICICIBANK"]
correlation = returns.corr().loc["SBIN", "ICICIBANK"]

weights = np.linspace(0, 1, 101)
frontier = []
for w_sbin in weights:
    w_icici = 1 - w_sbin
    ret = w_sbin * mu["SBIN"] + w_icici * mu["ICICIBANK"]
    vol = np.sqrt(
        (w_sbin ** 2 * sigma["SBIN"] ** 2)
        + (w_icici ** 2 * sigma["ICICIBANK"] ** 2)
        + (2 * w_sbin * w_icici * covariance)
    )
    frontier.append({"Weight_SBIN": w_sbin, "Weight_ICICI": w_icici, "Daily_Vol": vol, "Daily_Return": ret})
frontier = pd.DataFrame(frontier)

limit_rows = []
for label, limit in RISK_LIMITS.items():
    feasible = frontier[frontier["Daily_Vol"] <= limit]
    if feasible.empty:
        best = None
        limit_rows.append({"Limit": label, "Vol_Limit": limit, "Feasible_Portfolios": 0, "Best_SBIN": np.nan, "Best_ICICI": np.nan, "Best_Return": np.nan, "Best_Vol": np.nan})
    else:
        best = feasible.loc[feasible["Daily_Return"].idxmax()]
        limit_rows.append({"Limit": label, "Vol_Limit": limit, "Feasible_Portfolios": len(feasible), "Best_SBIN": best.Weight_SBIN, "Best_ICICI": best.Weight_ICICI, "Best_Return": best.Daily_Return, "Best_Vol": best.Daily_Vol})
limit_summary = pd.DataFrame(limit_rows)
min_vol = frontier.loc[frontier["Daily_Vol"].idxmin()]

print("Mean daily log returns:")
display(mu.to_frame("mean"))
print("Sample daily volatility:")
display(sigma.to_frame("stdev"))
print(f"Covariance={covariance:.8f}; correlation={correlation:.4f}")
print("Minimum-volatility sampled portfolio:")
display(min_vol.to_frame().T)
print("Best feasible portfolio under each source's limit:")
display(limit_summary)

Mean daily log returns:


,mean
SBIN,0.001218
ICICIBANK,0.001835


Sample daily volatility:


,stdev
SBIN,0.016094
ICICIBANK,0.010450


Covariance=0.00006864; correlation=0.4081
Minimum-volatility sampled portfolio:


,Weight_SBIN,Weight_ICICI,Daily_Vol,Daily_Return
18,0.18,0.82,0.010103,0.001724


Best feasible portfolio under each source's limit:


,Limit,Vol_Limit,Feasible_Portfolios,Best_SBIN,Best_ICICI,Best_Return,Best_Vol
0,transcript_1_03pct,0.01030,26,0.05,0.95,0.001804,0.010282
1,transcript_1_303pct,0.01303,72,0.00,1.00,0.001835,0.010450
2,slides_2_7pct,0.02700,101,0.00,1.00,0.001835,0.010450


## 6. Chart the efficient-frontier source difference

The vertical lines show why the two limits should both be documented. A tighter limit changes the portfolio answer; a loose `2.7%` limit makes the all-ICICI portfolio feasible and therefore maximizes historical return among the two assets.

In [18]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ax.plot(frontier["Daily_Vol"] * 100, frontier["Daily_Return"] * 100, color="#2563eb", lw=2.2, label="101 sampled portfolios")
ax.scatter(min_vol["Daily_Vol"] * 100, min_vol["Daily_Return"] * 100, s=80, color="#f59e0b", zorder=5, label="minimum volatility")
colors = {"transcript_1_03pct": "#dc2626", "transcript_1_303pct": "#7c3aed", "slides_2_7pct": "#16a34a"}
for _, row in limit_summary.iterrows():
    ax.axvline(row["Vol_Limit"] * 100, color=colors[row["Limit"]], linestyle="--", alpha=0.75, label=row["Limit"].replace("_", " "))
    if not np.isnan(row["Best_Vol"]):
        ax.scatter(row["Best_Vol"] * 100, row["Best_Return"] * 100, s=65, color=colors[row["Limit"]], edgecolor="black", linewidth=0.5)
ax.set_xlabel("Daily portfolio volatility (%)")
ax.set_ylabel("Mean daily log return (%)")
ax.set_title("SBIN/ICICI Frontier Under Transcript and Slide Risk Limits")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
out = DATA_DIR / "chart_addendum_1_sfm03_limit_comparison.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Saved chart_addendum_1_sfm03_limit_comparison.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\3558559115.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Official practice-exercise formula map

The practice exercise pack shifts from the lecture's SBIN/ICICI/NIFTY files to a Stooq workflow with `HDB`, `SBIN`, and `NIFTY`. The formulas are basic but important because they make the Excel answer cells auditable.

In [19]:
formula_map = pd.DataFrame([
    {"Task": "Daily NIFTY return", "Excel formula idea": "=(today - yesterday) / yesterday", "Python equivalent": "pct_change()"},
    {"Task": "Minimum SBIN price", "Excel formula idea": "=MIN(SBIN_range)", "Python equivalent": "sbin.min()"},
    {"Task": "Date of SBIN minimum", "Excel formula idea": "=XLOOKUP(min_price, SBIN_range, Date_range)", "Python equivalent": "idxmin() then Date lookup"},
    {"Task": "Mean price", "Excel formula idea": "=AVERAGE(price_range)", "Python equivalent": "mean()"},
    {"Task": "Sample standard deviation", "Excel formula idea": "=STDEV.S(range)", "Python equivalent": "std(ddof=1)"},
    {"Task": "Variance from stdev", "Excel formula idea": "=stdev_cell^2", "Python equivalent": "sigma**2"},
    {"Task": "10-day Monte Carlo", "Excel formula idea": "=previous_price*EXP(NORM.INV(RAND(), mean, std))", "Python equivalent": "price[t-1] * exp(normal(mean, std))"},
])
display(formula_map)
print("Official worked answer key values used for Q2-Q10:")
display(answer_key)

,Task,Excel formula idea,Python equivalent
0,Daily NIFTY return,=(today - yesterday) / yesterday,pct_change()
1,Minimum SBIN price,=MIN(SBIN_range),sbin.min()
2,Date of SBIN minimum,"=XLOOKUP(min_price, SBIN_range, Date_range)",idxmin() then Date lookup
3,Mean price,=AVERAGE(price_range),mean()
4,Sample standard deviation,=STDEV.S(range),std(ddof=1)
5,Variance from stdev,=stdev_cell^2,sigma**2
6,10-day Monte Carlo,"=previous_price*EXP(NORM.INV(RAND(), mean, std))","price[t-1] * exp(normal(mean, std))"


Official worked answer key values used for Q2-Q10:


,Question_ID,Question,Official_Worked_Answer
0,Q2,Identify the minimum price of SBIN using MIN().,62.27
1,Q3,Return the date corresponding to the minimum p...,2025-07-09 00:00:00
2,Q4,Compute the mean price of SBIN using AVERAGE().,69.7967088607595
3,Q5,Compute the mean price of HDB using AVERAGE().,35.61459556962026
4,Q6,Compute the sample standard deviation of SBIN ...,4.810624447070153
5,Q7,Compute the sample standard deviation of HDB p...,1.702721402547587
6,Q8,Compute the standard deviation of daily return...,0.007667194761460805
7,Q9,Compute the average of daily returns of NIFTY ...,-0.0001291406931850168
8,Q10,Calculate the variance of daily returns using ...,5.8785875510172e-05


## 8. Practice Monte Carlo with the official answer-key mean and volatility

The official practice workbook supplies the NIFTY daily-return mean and sample standard deviation. This section uses those answer-key values in the same `EXP(NORM.INV(RAND(), mean, std))` structure, with a fixed seed and an editable start price.

In [20]:
mean_return = float(answer_key.loc[answer_key["Question_ID"] == "Q9", "Official_Worked_Answer"].iloc[0])
stdev_return = float(answer_key.loc[answer_key["Question_ID"] == "Q8", "Official_Worked_Answer"].iloc[0])
variance_return = float(answer_key.loc[answer_key["Question_ID"] == "Q10", "Official_Worked_Answer"].iloc[0])
print(f"Practice answer-key mean={mean_return:.9f}, stdev={stdev_return:.9f}, variance={variance_return:.9e}")
print(f"Check stdev^2={stdev_return**2:.9e}")

rng = np.random.default_rng(SEED)
shocks = rng.normal(mean_return, stdev_return, size=(DAYS, N_SIM))
paths = np.empty((DAYS + 1, N_SIM))
paths[0] = MONTE_CARLO_START_PRICE
for day in range(1, DAYS + 1):
    paths[day] = paths[day - 1] * np.exp(shocks[day - 1])
final_prices = paths[-1]
mc_summary = pd.DataFrame({
    "Metric": ["start price", "mean final", "median final", "5th percentile VaR floor", "expected shortfall below 5th", "95th percentile"],
    "Value": [MONTE_CARLO_START_PRICE, final_prices.mean(), np.median(final_prices), np.percentile(final_prices, 5), final_prices[final_prices <= np.percentile(final_prices, 5)].mean(), np.percentile(final_prices, 95)],
})
display(mc_summary)

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.6))
for i in range(80):
    axes[0].plot(range(DAYS + 1), paths[:, i], color="#2563eb", alpha=0.12, linewidth=0.8)
axes[0].plot(range(DAYS + 1), np.percentile(paths, 50, axis=1), color="#111827", linewidth=2.0, label="median path")
axes[0].set_title("10-Day Practice MC Paths")
axes[0].set_xlabel("Trading day")
axes[0].set_ylabel("Simulated price")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

var5 = np.percentile(final_prices, 5)
es5 = final_prices[final_prices <= var5].mean()
axes[1].hist(final_prices, bins=50, color="#0f766e", alpha=0.75)
axes[1].axvline(var5, color="#dc2626", linestyle="--", label=f"P5={var5:,.0f}")
axes[1].axvline(es5, color="#7c2d12", linestyle=":", label=f"ES={es5:,.0f}")
axes[1].set_title("Final Price Distribution")
axes[1].set_xlabel("Day-10 price")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
fig.tight_layout()
out = DATA_DIR / "chart_addendum_3_sfm03_practice_mc.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Practice answer-key mean=-0.000129141, stdev=0.007667195, variance=5.878587551e-05
Check stdev^2=5.878587551e-05


,Metric,Value
0,start price,18065.000000
1,mean final,18052.772745
2,median final,18050.969135
3,5th percentile VaR floor,17348.373140
4,expected shortfall below 5th,17164.628224
5,95th percentile,18783.016803


Saved chart_addendum_3_sfm03_practice_mc.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23796\2210990872.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Bollinger formulas and breach count

The official notes define Bollinger Bands as a 20-period moving average plus or minus two 20-period sample standard deviations. This code confirms the formula on the lecture NIFTY data and counts how often the close falls outside the band.

In [21]:
bb = nifty.set_index("Date").copy()
bb["SMA20"] = bb["NIFTY"].rolling(20).mean()
bb["STD20"] = bb["NIFTY"].rolling(20).std(ddof=1)
bb["Upper"] = bb["SMA20"] + 2 * bb["STD20"]
bb["Lower"] = bb["SMA20"] - 2 * bb["STD20"]
bb["Outside"] = (bb["NIFTY"] > bb["Upper"]) | (bb["NIFTY"] < bb["Lower"])
valid = bb.dropna()
breach_rate = valid["Outside"].mean()
print(f"Bollinger valid rows={len(valid)}, outside-band observations={valid['Outside'].sum()}, outside-band rate={breach_rate:.2%}")
display(valid.tail()[["NIFTY", "SMA20", "STD20", "Upper", "Lower", "Outside"]])

Bollinger valid rows=2436, outside-band observations=223, outside-band rate=9.15%


,NIFTY,SMA20,STD20,Upper,Lower,Outside
Date,,,,,,
2023-04-24,17743.400391,17453.517676,307.081788,18067.681252,16839.354099,False
2023-04-25,17769.250000,17484.385156,306.194224,18096.773605,16871.996708,False
2023-04-26,17813.599609,17521.220117,298.817423,18118.854963,16923.585272,False
2023-04-27,17915.050781,17569.720117,278.400665,18126.521448,17012.918787,False
2023-04-28,18065.000000,17623.685156,263.439689,18150.564534,17096.805779,False


## 10. Addendum checklist

Covered here: official Bessel workbook numbers, source-difference risk limits, Stooq practice answer key, formula map, reproducible 10-day Monte Carlo, and exact Bollinger formulas. The original notebook remains preserved.